# MeloTTS Text-to-Speech with OpenVINO

This notebook summarizes the MeloTTS work from the Automotive.AI.models project and demonstrates a complete workflow: model download, PyTorch-to-OpenVINO conversion, and OpenVINO inference benchmark.

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/melo-tts/melo-tts.ipynb" />

[MeloTTS](https://github.com/myshell-ai/MeloTTS) is a multilingual TTS system based on VITS. In this project, the PyTorch pipeline is split into OpenVINO-friendly subgraphs and exported into two IR models:

- `melotts_enc.xml`: speaker embedding + text encoder + duration predictors (`sdp` and `dp`)
- `melotts_dec.xml`: flow (reverse) + HiFiGAN generator

Dynamic, data-dependent logic (length regulator / `generate_path`) is kept in Python for flexibility.

All scripts and the `melo_torch` / `melo_openvino` packages are bundled in this notebook folder, so it runs standalone without any external local paths.

#### Table of contents
- [Prerequisites](#Prerequisites)
- [Install Dependencies](#Install-Dependencies)
- [Download Checkpoints](#Download-Checkpoints)
- [Convert Models to OpenVINO IR](#Convert-Models-to-OpenVINO-IR)
- [Run Inference and Benchmark](#Run-Inference-and-Benchmark)
- [Listen to Output Samples](#Listen-to-Output-Samples)
- [Interactive demo](#Interactive-demo)

### Installation Instructions
This is a self-contained example that relies solely on its own bundled code.

⚠️ **EXPERIMENTAL NOTEBOOK**
This notebook demonstrates a custom integration that may require environment-specific adjustments.

## Prerequisites
[back to top ⬆️](#Table-of-contents)

In [1]:
import requests
from pathlib import Path

for fname in ("notebook_utils.py", "pip_helper.py"):
    if not Path(fname).exists():
        r = requests.get(
            url=f"https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/{fname}",
        )
        Path(fname).write_text(r.text)

from notebook_utils import collect_telemetry

collect_telemetry("melo-tts.ipynb")

# All bundled scripts and packages live next to this notebook.
project_dir = Path.cwd()

## Install Dependencies
[back to top ⬆️](#Table-of-contents)

The conversion and inference scripts depend on PyTorch + OpenVINO stacks. This cell installs the bundled dependency file.

In [2]:
from pip_helper import pip_install

pip_install("-q", "-r", str(project_dir / "requirements.txt"))

## Download Checkpoints
[back to top ⬆️](#Table-of-contents)

In [3]:
from download import download

download(lang=["ZH"])

MeloTTS 下载脚本
目标语言: ZH

[1/3] 检查 unidic 字典...
  ✓ unidic 字典已存在: /home/anna/WorkSpace/LIBRARIES/install/miniforge3/envs/melotts/lib/python3.11/site-packages/unidic/dicdir

[2/2] 下载模型...


/home/anna/WorkSpace/LIBRARIES/install/miniforge3/envs/melotts/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



  [ZH] TTS 模型已存在，跳过下载: /mnt/workspace/Projects/openvino_notebooks/notebooks/melo-tts/models/MeloTTS-Chinese
  [ZH] BERT (bert-base-multilingual-uncased) 已存在，跳过下载: bert/bert-base-multilingual-uncased/
  [ZH] BERT (hfl/chinese-roberta-wwm-ext-large) 已存在，跳过下载: bert/hfl--chinese-roberta-wwm-ext-large/

下载完成！现在可以运行: python inference-torch.py


## Convert Models to OpenVINO IR
[back to top ⬆️](#Table-of-contents)

The conversion script exports two IR models and keeps dynamic length regulation in Python runtime.

- Input language: `ZH`
- Output directory: `models/OpenVINO/MeloTTS-Chinese`

In [4]:
from model_convert import convert

convert("ZH", "./models/OpenVINO/MeloTTS-Chinese")

/home/anna/WorkSpace/LIBRARIES/install/miniforge3/envs/melotts/lib/python3.11/site-packages/librosa/util/files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


OpenVINO IR 已存在，跳过转换:
      ./models/OpenVINO/MeloTTS-Chinese/melotts_enc.xml
      ./models/OpenVINO/MeloTTS-Chinese/melotts_dec.xml
      BERT hfl--chinese-roberta-wwm-ext-large: 已存在，跳过
      BERT bert-base-multilingual-uncased: 已存在，跳过
转换完成（复用已有 IR）。


## Run Inference and Benchmark
[back to top ⬆️](#Table-of-contents)

Run both baselines from the project scripts:
- `inference_torch.py`
- `inference_ov.py`

Each script prints total inference time, audio duration, and average RTF.

In [5]:
import os
import time
import shutil

import soundfile as sf

LANG = "ZH"
speed = 1.0

# Text batch to synthesize (mix of Chinese and English)
syn_text_batch = [
    "我最近在学习machine learning，希望能够在未来的artificial intelligence领域有所建树。",
    "Good one. Okay, fine, I'm just gonna leave this sock monkey here. Goodbye.",
    "Hello! Welcome to the MeloTTS demo with OpenVINO acceleration.",
    "其实我真的有发现，我是一个特别善于观察别人情绪的人。",
    "如果祖国需要，请把我埋在遥远的山岗，让我的身躯长成一道无形的屏障，往来的战友会为我泪落两行",
    "如果祖国需要，请让我紧握滚烫的钢枪，让我的双手握出一轮沧桑的红日，冰冷的大地会为我抚慰创伤",
    "如果祖国需要，请让更多的人走向杀敌的战场，让我和你的心房 在蓝天上，跳跃出永远不朽的乐章 。",
]


def benchmark(model, speaker_id, output_dir, texts, speed=1.0):
    """Run inference on a batch of texts and report per-item and average RTF."""
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    # Warmup so first-run initialization overhead does not skew the RTF measurement
    print("Warmup...")
    model.tts_to_file("你好，世界。", speaker_id, os.path.join(output_dir, "warmup.wav"), speed=speed, quiet=True)
    print("Warmup done\n")

    total_audio_duration = 0.0
    total_infer_time = 0.0

    for i, text in enumerate(texts):
        output_path = os.path.join(output_dir, f"output_{i}.wav")

        start_time = time.time()
        model.tts_to_file(text, speaker_id, output_path, speed=speed, quiet=True)
        infer_time = time.time() - start_time

        audio_data, sr = sf.read(output_path)
        audio_duration = len(audio_data) / sr

        total_audio_duration += audio_duration
        total_infer_time += infer_time

        rtf = infer_time / audio_duration
        print(f"  [{i+1}/{len(texts)}] infer: {infer_time:.2f}s | audio: {audio_duration:.2f}s | RTF: {rtf:.3f}")

    print(f"\n{'='*50}")
    print(f"Total inference time: {total_infer_time:.2f}s")
    print(f"Total audio duration: {total_audio_duration:.2f}s")
    print(f"Average RTF: {total_infer_time / total_audio_duration:.3f}")
    print(f"  RTF < 1.0 means faster than real time; lower is better")
    print(f"{'='*50}")

In [6]:
from melo_torch.api import TTS as TorchTTS

torch_device = "cpu"
print(f"===== PyTorch-{torch_device} inference =====")

torch_model_dir = project_dir / "models" / "MeloTTS-Chinese"
torch_bert_dir = torch_model_dir / "bert"

torch_model = TorchTTS(
    language=LANG,
    device=torch_device,
    config_path=str(torch_model_dir / "config.json"),
    ckpt_path=str(torch_model_dir / "checkpoint.pth"),
    bert_dir=str(torch_bert_dir) if torch_bert_dir.is_dir() else None,
)

benchmark(
    torch_model,
    torch_model.hps.data.spk2id[LANG],
    str(project_dir / "test_output_torch"),
    syn_text_batch,
    speed=speed,
)

===== PyTorch-cpu inference =====


/home/anna/WorkSpace/LIBRARIES/install/miniforge3/envs/melotts/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache


Warmup...


Loading model cost 0.300 seconds.
Prefix dict has been built successfully.
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 2621.00it/s, Materializing param=cls.predictions.transform.dense.weight]                 


Warmup done

  [1/7] infer: 1.40s | audio: 6.32s | RTF: 0.222
  [2/7] infer: 1.37s | audio: 6.07s | RTF: 0.225
  [3/7] infer: 1.19s | audio: 4.71s | RTF: 0.252
  [4/7] infer: 0.94s | audio: 4.31s | RTF: 0.218
  [5/7] infer: 1.75s | audio: 8.01s | RTF: 0.219
  [6/7] infer: 1.82s | audio: 8.51s | RTF: 0.214
  [7/7] infer: 1.72s | audio: 8.02s | RTF: 0.215

Total inference time: 10.19s
Total audio duration: 45.95s
Average RTF: 0.222
  RTF < 1.0 means faster than real time; lower is better


In [7]:
from melo_openvino.api import TTS as OVTTS

print("===== OpenVINO inference =====")

ov_model_dir = project_dir / "models" / "OpenVINO" / "MeloTTS-Chinese"

ov_model = OVTTS(language=LANG, device="GPU", ov_model_dir=str(ov_model_dir))

benchmark(
    ov_model,
    ov_model.hps.data.spk2id[LANG],
    str(project_dir / "test_output_ov"),
    syn_text_batch,
    speed=speed,
)

===== OpenVINO inference =====
Warmup...
Warmup done

  [1/7] infer: 0.79s | audio: 6.47s | RTF: 0.122
  [2/7] infer: 0.82s | audio: 5.98s | RTF: 0.138
  [3/7] infer: 0.55s | audio: 4.69s | RTF: 0.117
  [4/7] infer: 0.51s | audio: 4.28s | RTF: 0.118
  [5/7] infer: 1.04s | audio: 8.24s | RTF: 0.126
  [6/7] infer: 1.06s | audio: 8.45s | RTF: 0.126
  [7/7] infer: 0.99s | audio: 8.22s | RTF: 0.120

Total inference time: 5.76s
Total audio duration: 46.33s
Average RTF: 0.124
  RTF < 1.0 means faster than real time; lower is better


## Listen to Output Samples
[back to top ⬆️](#Table-of-contents)

In [8]:
from pathlib import Path
from IPython.display import Audio, display

torch_out = project_dir / "test_output_torch" / "output_0.wav"
ov_out = project_dir / "test_output_ov" / "output_0.wav"

print("PyTorch sample:", torch_out)
if torch_out.exists():
    display(Audio(filename=str(torch_out)))
else:
    print("PyTorch output not found. Run inference cell first.")

print("OpenVINO sample:", ov_out)
if ov_out.exists():
    display(Audio(filename=str(ov_out)))
else:
    print("OpenVINO output not found. Run inference cell first.")

PyTorch sample: /mnt/workspace/Projects/openvino_notebooks/notebooks/melo-tts/test_output_torch/output_0.wav


OpenVINO sample: /mnt/workspace/Projects/openvino_notebooks/notebooks/melo-tts/test_output_ov/output_0.wav


## Interactive demo
[back to top ⬆️](#Table-of-contents)

Launch a Gradio interface to synthesize speech interactively with the OpenVINO model. Enter your text, choose a speaker and speed, then generate audio accelerated by OpenVINO Runtime.

In [9]:
from gradio_helper import make_demo

demo = make_demo(ov_model, LANG)

# if you are launching remotely, specify server_name and server_port
#  demo.launch(server_name='your server name', server_port='server port in int')
# if you have any issue to launch on your platform, you can pass share=True to launch method:
# demo.launch(share=True)
# it creates a publicly shareable link for the interface. Read more in the docs: https://gradio.app/docs/
try:
    demo.launch(debug=True)
except Exception:
    demo.launch(debug=True, share=True)

/mnt/workspace/Projects/openvino_notebooks/notebooks/melo-tts/gradio_helper.py:81: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=theme, css=css, title="MeloTTS with OpenVINO") as demo:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


/home/anna/WorkSpace/LIBRARIES/install/miniforge3/envs/melotts/lib/python3.11/site-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/home/anna/WorkSpace/LIBRARIES/install/miniforge3/envs/melotts/lib/python3.11/site-packages/gradio/processing_utils.py:724: UserWarning: Trying to convert audio automatically from float32 to 16-bit int format.
  warnings.warn(warning.format(data.dtype))


Keyboard interruption in main thread... closing server.


## Notes
- `inference_ov.py` currently uses CPU by default.
- The conversion script includes optional BERT conversion so the text frontend can also run with OpenVINO IR.
- For multi-language testing, change `--lang` in download/convert commands (for example: `EN`, `JP`, `FR`, `ES`, `KR`).